# High-Throughput Edge Telemetry & Schema-Enforced JSON Dispatcher with Nemotron-3 Nano (30B/3B MoE)

> **Google Cloud | Gemini Enterprise Agent Platform | NVIDIA Nemotron-3 Nano in Model Garden**

---

### 📖 Executive Overview
**NVIDIA Nemotron-3 Nano** utilizes a sparse **30B Total / 3B Active Mixture-of-Experts (MoE)** architecture. Activating only 3 billion parameters per token provides extreme inference throughput and low memory consumption on lightweight accelerators (NVIDIA L4 or single RTX PRO 6000).

In this notebook, you will build an **Edge IoT Anomaly Extraction & Event Router**:
1. **Ingest Industrial Sensory Telemetry**: Ingest unstructured log streams from turbine bearings, chemical pumps, and power inverters.
2. **Schema-Enforced JSON Extraction**: Enforce strict **Pydantic** typing for severity ratings, anomaly thresholds, and mitigation urgency.
3. **Automated Dispatch Routing**: Route validated events to simulated Google Cloud Pub/Sub topics and Cloud Function trigger endpoints.

> [!TIP]
> **Flexible Endpoint Operation Modes:**
> * **Mode 1 (`AUTO_DISCOVER`) [Default]**: Automatically scans and binds to active Model Garden deployments in your GCP project. (To deploy via UI ahead of time: [Vertex AI Model Garden](https://console.cloud.google.com/vertex-ai/model-garden), recommended profile: **g4-standard-48 or g2-standard-16**).
> * **Mode 2 (`USE_EXISTING_ENDPOINT`)**: Bind directly to any custom endpoint or fine-tuned model by setting `CUSTOM_ENDPOINT_ID`.
> * **Mode 3 (`CREATE_CUSTOM_ENDPOINT`)**: Programmatically create a new Vertex AI Endpoint and deploy your custom container or model weights directly from the notebook.

---

### 📋 Prerequisites & Setup
* Target Model: **Nemotron-3 Nano** (`nemotron-3-nano-30b-a3b-bf16` or `fp8`) or custom endpoint.
* Recommended Accelerator: `g4-standard-48` (1x RTX PRO 6000) or `g2-standard-16` (1x L4).


### Step 1: Harmonized Dependency Installation


In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import sys
import subprocess

# Harmonized dependency installation with conflict prevention
!pip install --quiet --no-warn-conflicts "google-cloud-aiplatform>=1.70.0" "openai>=1.50.0,<2.0.0" "pydantic>=2.0.0,<3.0.0" "rich>=13.7.0,<14.0.0" "requests>=2.31.0,<=2.32.4" "protobuf>=3.20.2,<5.0.0dev"

print("✓ Harmonized dependencies installed successfully.")


### Step 2: Environment Configuration & Endpoint Operation Selector

> [!TIP]
> **Flexible Endpoint Operation Modes:**
> * **Mode 1 (`AUTO_DISCOVER`) [Default]**: Automatically scans and binds to active Model Garden deployments in your GCP project. (To deploy via UI ahead of time: [Vertex AI Model Garden](https://console.cloud.google.com/vertex-ai/model-garden), recommended profile: **g4-standard-48 or g2-standard-16**).
> * **Mode 2 (`USE_EXISTING_ENDPOINT`)**: Bind directly to any custom endpoint or fine-tuned model by setting `CUSTOM_ENDPOINT_ID`.
> * **Mode 3 (`CREATE_CUSTOM_ENDPOINT`)**: Programmatically create a new Vertex AI Endpoint and deploy your custom container or model weights directly from the notebook.


In [ ]:
import os
import subprocess
from google.cloud import aiplatform
from rich.console import Console
from rich.table import Table
from rich.panel import Panel

console = Console()

# ==============================================================================
# Step 2: Environment Configuration & Endpoint Operation Selector
# ==============================================================================
# Choose your connection/deployment operation:
# 1. "AUTO_DISCOVER" : (Default) Auto-detects and binds to active Model Garden endpoints.
# 2. "USE_EXISTING_ENDPOINT": Binds directly to your custom or existing endpoint ID/Name.
# 3. "CREATE_CUSTOM_ENDPOINT": Programmatically creates a new Vertex AI endpoint and
#                              deploys a custom container/model artifact.
# ==============================================================================

PROJECT_ID = ""                # @param {type:"string"} - Set your GCP Project ID (Leave blank to auto-detect)
REGION = "us-central1"         # @param ["us-central1", "us-east4", "us-west1", "europe-west4"] {allow-input: true}
OPERATION_MODE = "AUTO_DISCOVER" # @param ["AUTO_DISCOVER", "USE_EXISTING_ENDPOINT", "CREATE_CUSTOM_ENDPOINT"]

# --- Mode: USE_EXISTING_ENDPOINT Settings ---
CUSTOM_ENDPOINT_ID = ""        # @param {type:"string"} - e.g. "1234567890" or "projects/.../endpoints/..."

# --- Mode: CREATE_CUSTOM_ENDPOINT Settings ---
CUSTOM_ENDPOINT_DISPLAY_NAME = "nano-custom-endpoint" # @param {type:"string"}
CUSTOM_SERVING_CONTAINER_URI = "us-docker.pkg.dev/vertex-ai/vertex-vision-model-garden-dockers/vllm-serve:latest" # @param {type:"string"}
CUSTOM_ARTIFACT_URI = ""       # @param {type:"string"} - Optional: Cloud Storage path (e.g. gs://your-bucket/model-weights)
CUSTOM_MACHINE_TYPE = "g4-standard-48" # @param ["g4-standard-48", "g4-standard-384", "g2-standard-16", "g2-standard-96", "a4-highgpu-8g", "a3-ultragpu-8g"] {allow-input: true}
CUSTOM_ACCELERATOR_TYPE = "NVIDIA_RTX_PRO_6000" # @param ["NVIDIA_RTX_PRO_6000", "NVIDIA_L4", "NVIDIA_B200", "NVIDIA_H200"] {allow-input: true}
CUSTOM_ACCELERATOR_COUNT = 1   # @param {type:"integer"}

# 1. Resolve GCP Project ID
if not PROJECT_ID.strip():
    try:
        PROJECT_ID = subprocess.check_output(
            ["gcloud", "config", "get-value", "project"], 
            stderr=subprocess.DEVNULL
        ).decode().strip()
    except Exception:
        PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "cpe-slarbi-nvd-ant-demos")

console.print(f"[bold green]✓ GCP Project:[/bold green] [cyan]{PROJECT_ID}[/cyan] | [bold green]Region:[/bold green] [cyan]{REGION}[/cyan] | [bold green]Mode:[/bold green] [yellow]{OPERATION_MODE}[/yellow]")
aiplatform.init(project=PROJECT_ID, location=REGION)

# 2. Unified Endpoint Resolver & Deployer
def resolve_or_create_endpoint(
    project_id: str, 
    location: str, 
    target_keywords: list,
    mode: str = "AUTO_DISCOVER",
    custom_endpoint_id: str = "",
    create_params: dict = None
) -> aiplatform.Endpoint:
    # -------------------------------------------------------------------------
    # Path 1: Connect to an Existing Custom Endpoint
    # -------------------------------------------------------------------------
    if mode == "USE_EXISTING_ENDPOINT" or (custom_endpoint_id and custom_endpoint_id.strip()):
        ep_name = custom_endpoint_id.strip()
        if not ep_name:
            raise ValueError("OPERATION_MODE is 'USE_EXISTING_ENDPOINT', but CUSTOM_ENDPOINT_ID is empty.")
        
        ep_path = ep_name if ep_name.startswith("projects/") else f"projects/{project_id}/locations/{location}/endpoints/{ep_name}"
        ep = aiplatform.Endpoint(ep_path)
        console.print(Panel(
            f"[bold]Display Name:[/bold] {ep.display_name}\n"
            f"[bold]Endpoint ID:[/bold]  {ep.name.split('/')[-1]}\n"
            f"[bold]Resource:[/bold]     {ep.name}",
            title="✓ BOUND TO CUSTOM ENDPOINT",
            border_style="green"
        ))
        return ep

    # -------------------------------------------------------------------------
    # Path 2: Create & Deploy a Custom Endpoint on Demand
    # -------------------------------------------------------------------------
    if mode == "CREATE_CUSTOM_ENDPOINT":
        p = create_params or {}
        disp_name = p.get("display_name", "custom-nemotron-endpoint")
        container_uri = p.get("container_uri", "us-docker.pkg.dev/vertex-ai/vertex-vision-model-garden-dockers/vllm-serve:latest")
        artifact_uri = p.get("artifact_uri", "")
        mach_type = p.get("machine_type", "g4-standard-48")
        acc_t = p.get("accelerator_type", "NVIDIA_RTX_PRO_6000")
        acc_c = p.get("accelerator_count", 1)

        console.print(Panel(
            f"[bold]Endpoint Name:[/bold]     {disp_name}\n"
            f"[bold]Serving Container:[/bold] {container_uri}\n"
            f"[bold]Hardware Profile:[/bold]  {mach_type} ({acc_c}x {acc_t})",
            title="🚀 INITIATING CUSTOM ENDPOINT DEPLOYMENT",
            border_style="yellow"
        ))

        console.print("⏳ [1/3] Registering custom model with Vertex AI...")
        upload_kwargs = {
            "display_name": f"{disp_name}-model",
            "serving_container_image_uri": container_uri,
        }
        if artifact_uri.strip():
            upload_kwargs["artifact_uri"] = artifact_uri.strip()

        model_res = aiplatform.Model.upload(**upload_kwargs)
        console.print(f"✓ Model registered: [cyan]{model_res.resource_name}[/cyan]")

        console.print("⏳ [2/3] Creating dedicated Vertex AI endpoint...")
        custom_endpoint = aiplatform.Endpoint.create(display_name=disp_name)
        console.print(f"✓ Endpoint created: [cyan]{custom_endpoint.resource_name}[/cyan]")

        console.print("⏳ [3/3] Deploying model to endpoint (provisioning compute and loading weights)...")
        model_res.deploy(
            endpoint=custom_endpoint,
            machine_type=mach_type,
            accelerator_type=acc_t,
            accelerator_count=acc_c,
            traffic_percentage=100,
            sync=True
        )
        console.print(Panel(
            f"[bold]Display Name:[/bold] {custom_endpoint.display_name}\n"
            f"[bold]Endpoint ID:[/bold]  {custom_endpoint.name.split('/')[-1]}\n"
            f"[bold]Resource:[/bold]     {custom_endpoint.name}",
            title="✓ CUSTOM ENDPOINT DEPLOYED SUCCESSFULLY",
            border_style="bold green"
        ))
        return custom_endpoint

    # -------------------------------------------------------------------------
    # Path 3: Auto-Discovery of Active Model Garden Endpoints (Default)
    # -------------------------------------------------------------------------
    console.print(f"🔍 Scanning for active Vertex AI endpoints in [cyan]{project_id}[/cyan] ({location})...")
    endpoints = aiplatform.Endpoint.list(order_by="create_time desc")
    
    if not endpoints:
        console.print(Panel(
            f"No active endpoints found in project [cyan]{project_id}[/cyan] / [cyan]{location}[/cyan].\n\n"
            f"1. Open Model Garden: https://console.cloud.google.com/vertex-ai/model-garden\n"
            f"2. Or set OPERATION_MODE = 'CREATE_CUSTOM_ENDPOINT' to deploy directly.\n"
            f"3. Or set OPERATION_MODE = 'USE_EXISTING_ENDPOINT' with CUSTOM_ENDPOINT_ID.",
            title="⚠️ NO ACTIVE ENDPOINTS FOUND",
            border_style="bold red"
        ))
        raise RuntimeError("No active endpoints found. Please deploy a Model Garden model or select a custom mode.")

    table = Table(title=f"Active Endpoints in {project_id}", border_style="blue")
    table.add_column("#", style="dim", width=4)
    table.add_column("Display Name", style="bold white")
    table.add_column("Endpoint ID", style="cyan")
    
    for idx, ep in enumerate(endpoints, start=1):
        table.add_row(str(idx), ep.display_name, ep.name.split("/")[-1])
    console.print(table)

    # 1st Priority: Match target model keywords
    matching = [
        ep for ep in endpoints 
        if any(k.lower() in (ep.display_name or "").lower() for k in target_keywords)
    ]

    if matching:
        selected = matching[0]
        console.print(Panel(
            f"[bold]Attached Model:[/bold]   {selected.display_name}\n"
            f"[bold]Endpoint ID:[/bold]      {selected.name.split('/')[-1]}\n"
            f"[bold]Resource Path:[/bold]    {selected.name}",
            title=f"✓ AUTO-ATTACHED: Nemotron-3 Nano (30B/3B)",
            border_style="bold green"
        ))
        return selected

    # 2nd Priority: Fallback to any active Nemotron / NVIDIA endpoint
    generic_nemotron = [
        ep for ep in endpoints 
        if any(k in (ep.display_name or "").lower() for k in ["nemotron", "nvidia"])
    ]
    if generic_nemotron:
        selected = generic_nemotron[0]
        console.print(Panel(
            f"[bold]Attached Model:[/bold]   {selected.display_name}\n"
            f"[bold]Endpoint ID:[/bold]      {selected.name.split('/')[-1]}\n"
            f"[bold]Resource Path:[/bold]    {selected.name}",
            title="💡 AUTO-ATTACHED TO ACTIVE NEMOTRON ENDPOINT",
            border_style="bold yellow"
        ))
        return selected

    # 3rd Priority: Fallback to first available active endpoint
    selected = endpoints[0]
    console.print(f"💡 [dim]Fallback: Binding to active endpoint '{selected.display_name}' (ID: {selected.name.split('/')[-1]}).[/dim]")
    return selected

target_endpoint = resolve_or_create_endpoint(
    PROJECT_ID, 
    REGION, 
    target_keywords=["nano", "nemotron-3-nano", "nemotron", "nvidia"],
    mode=OPERATION_MODE,
    custom_endpoint_id=CUSTOM_ENDPOINT_ID,
    create_params={
        "display_name": CUSTOM_ENDPOINT_DISPLAY_NAME,
        "container_uri": CUSTOM_SERVING_CONTAINER_URI,
        "artifact_uri": CUSTOM_ARTIFACT_URI,
        "machine_type": CUSTOM_MACHINE_TYPE,
        "accelerator_type": CUSTOM_ACCELERATOR_TYPE,
        "accelerator_count": CUSTOM_ACCELERATOR_COUNT
    }
)


### Step 3: Define Pydantic Event Schema & Dispatch Rules
Defines the strictly-typed event model that Nemotron-3 Nano must extract.


In [ ]:
import json
from typing import Literal, List
from pydantic import BaseModel, Field, ValidationError

class TelemetryEvent(BaseModel):
    device_id: str = Field(description="Unique hardware identifier of the edge device")
    facility_zone: str = Field(description="Physical location or cluster sector")
    metric_name: str = Field(description="Target telemetry metric name (e.g. CORE_TEMP, VIBRATION_HZ)")
    recorded_value: float = Field(description="Numeric value recorded")
    unit: str = Field(description="Unit of measurement")
    anomaly_status: Literal["CRITICAL", "WARNING", "NORMAL"] = Field(description="Severity classification")
    dispatch_action: Literal["EMERGENCY_SHUTDOWN", "DISPATCH_FIELD_TECH", "THROTTLE_LOAD", "LOG_ONLY"] = Field(description="Automated mitigation route")
    confidence_score: float = Field(ge=0.0, le=1.0, description="Model classification confidence")
    reasoning: str = Field(description="Concise justification for the action")

console.print("[bold green]✓ Pydantic Telemetry Schema compiled successfully.[/bold green]")


### Step 4: Ingest Heterogeneous Industrial Telemetry Stream


In [ ]:
RAW_TELEMETRY_SAMPLES = [
    "DEV-ID: TURBINE-94B | LOC: Plant-North-Zone3 | SENSOR: Bearing Temp read 118.4 C (Threshold: 85 C). Rapid acceleration of heat curve detected. Risk of bearing seizure.",
    "DEV-ID: PUMP-12A | LOC: Refinery-Sector-7 | SENSOR: Flow rate 42.1 L/min (Nominal: 45 L/min). Minor cavitation hum noticed. Standard baseline variation.",
    "DEV-ID: INVERTER-08 | LOC: Solar-Array-West | SENSOR: Voltage fluctuation spikes at 840V DC (Safe Max: 750V DC). Inverter cooling fan failed to spin."
]

console.print(f"[bold cyan]Loaded {len(RAW_TELEMETRY_SAMPLES)} industrial telemetry samples.[/bold cyan]")


### Step 5: Execute High-Speed Schema Extraction & Event Routing (Rich Table Dashboard & Max Tokens)
Nemotron-3 Nano extracts structured JSON and validates against our Pydantic schema in real-time, rendered in a formatted event table. `max_tokens=1024` ensures that internal reasoning tokens never cause premature payload cutoff.


In [ ]:
import json
import re

def extract_clean_content(prediction_obj) -> str:
    """
    Extracts purely the clean assistant text from any Vertex AI / vLLM / OpenAI response format,
    filtering out raw API dictionaries, metadata envelopes, usage stats, and unicode artifacts.
    """
    if isinstance(prediction_obj, list) and len(prediction_obj) > 0:
        prediction_obj = prediction_obj[0]

    if isinstance(prediction_obj, str):
        try:
            prediction_obj = json.loads(prediction_obj)
        except Exception:
            pass

    content = ""
    if isinstance(prediction_obj, dict):
        # 1. Check for OpenAI/vLLM 'choices' format
        choices = prediction_obj.get("choices", [])
        if isinstance(choices, list) and len(choices) > 0:
            first_choice = choices[0]
            if isinstance(first_choice, dict):
                msg = first_choice.get("message", {})
                if isinstance(msg, dict):
                    content = msg.get("content", "")
                    if not content and "reasoning" in msg:
                        content = msg.get("reasoning", "")
                    if not content and "reasoning_content" in msg:
                        content = msg.get("reasoning_content", "")
                elif isinstance(msg, str):
                    content = msg
                if not content:
                    content = first_choice.get("text", "")
            elif isinstance(first_choice, str):
                content = first_choice

        # 2. Check for standard 'content', 'text', or 'predictions'
        if not content:
            content = prediction_obj.get("content", prediction_obj.get("text", ""))

        # 3. Direct message dict check
        if not content and "role" in prediction_obj and "content" in prediction_obj:
            content = prediction_obj.get("content", "")

    elif isinstance(prediction_obj, str):
        content = prediction_obj

    if not content:
        content = str(prediction_obj)

    # Normalize unicode spacing artifacts ( ,  )
    cleaned = str(content).replace("\u202f", " ").replace("\u00a0", " ").strip()
    return cleaned

def extract_json_from_text(raw_text: str) -> dict:
    """
    Robustly extracts and parses a JSON dictionary from LLM output,
    handling markdown blocks, preambles, reasoning wrappers, and trailing commas.
    """
    text = raw_text.strip()
    try:
        return json.loads(text)
    except Exception:
        pass

    if "```json" in text:
        content = text.split("```json", 1)[1]
        if "```" in content:
            content = content.split("```", 1)[0]
        try:
            return json.loads(content.strip())
        except Exception:
            sanitized = re.sub(r',\s*([\}\]])', r'\1', content.strip())
            try:
                return json.loads(sanitized)
            except Exception:
                pass

    if "```" in text:
        content = text.split("```", 1)[1]
        if "```" in content:
            content = content.split("```", 1)[0]
        try:
            return json.loads(content.strip())
        except Exception:
            sanitized = re.sub(r',\s*([\}\]])', r'\1', content.strip())
            try:
                return json.loads(sanitized)
            except Exception:
                pass

    first_brace = text.find("{")
    last_brace = text.rfind("}")
    if first_brace != -1 and last_brace != -1 and last_brace > first_brace:
        candidate = text[first_brace:last_brace + 1]
        try:
            return json.loads(candidate)
        except Exception:
            sanitized = re.sub(r',\s*([\}\]])', r'\1', candidate)
            try:
                return json.loads(sanitized)
            except Exception:
                pass

    raise ValueError(f"No valid JSON object found in model output: {text[:200]}")

def extract_and_route(raw_log: str) -> TelemetryEvent:
    schema_str = json.dumps(TelemetryEvent.model_json_schema(), indent=2)
    prompt = (
        f"You are an edge telemetry parser powered by NVIDIA Nemotron-3 Nano.\n"
        f"Extract the following raw log into a valid JSON object strictly matching this schema:\n{schema_str}\n\n"
        f"RAW LOG:\n{raw_log}\n\n"
        f"OUTPUT ONLY VALID JSON:"
    )

    payload = {
        "instances": [
            {
                "@requestFormat": "chatCompletions",
                "messages": [{"role": "user", "content": prompt}],
                "max_tokens": 1024,
                "temperature": 0.0
            }
        ]
    }

    try:
        res = target_endpoint.predict(instances=payload["instances"])
        text = extract_clean_content(res.predictions)
    except Exception:
        res = target_endpoint.predict(instances=[{"prompt": prompt, "max_tokens": 1024}])
        text = extract_clean_content(res.predictions)

    event_dict = extract_json_from_text(text)
    return TelemetryEvent.model_validate(event_dict)

event_table = Table(title="⚡ Edge Telemetry Structured Dispatch Dashboard", border_style="cyan")
event_table.add_column("Device ID", style="bold cyan")
event_table.add_column("Metric & Value", style="white")
event_table.add_column("Severity", style="bold")
event_table.add_column("Dispatch Route", style="bold magenta")
event_table.add_column("Confidence", style="yellow")
event_table.add_column("Reasoning", style="dim white")

for i, sample in enumerate(RAW_TELEMETRY_SAMPLES, 1):
    try:
        event = extract_and_route(sample)
        sev_style = "red" if event.anomaly_status == "CRITICAL" else ("yellow" if event.anomaly_status == "WARNING" else "green")
        event_table.add_row(
            event.device_id,
            f"{event.recorded_value} {event.unit} ({event.metric_name})",
            f"[{sev_style}]{event.anomaly_status}[/{sev_style}]",
            event.dispatch_action,
            f"{event.confidence_score*100:.0f}%",
            event.reasoning
        )
    except ValidationError as ve:
        console.print(f"[red]Validation Error on Sample {i}:[/red] {ve}")
    except Exception as e:
        console.print(f"[red]Processing Error on Sample {i}:[/red] {e}")

console.print(event_table)


### Step 6: Safe Teardown & Endpoint Lifecycle Management


In [ ]:
DELETE_SCRATCH_ENDPOINT = False  # @param {type:"boolean"}
if DELETE_SCRATCH_ENDPOINT and 'target_endpoint' in locals():
    console.print(f"[bold red]Cleaning up endpoint:[/bold red] {target_endpoint.display_name}...")
    target_endpoint.delete(force=True)
    console.print("[bold green]✓ Cleanup completed.[/bold green]")
else:
    console.print("[dim]Preserving endpoint for subsequent sessions.[/dim]")
